## Load Data and Packages

Auditing a recidivism prediction classifier for racial bias using Fairlearn, measuring demographic parity and equalized odds gaps across groups. First will load data and set up features.

In [ ]:
import pandas as pd
# Read the COMPAS data
df = pd.read_csv('../data/compas-scores-two-years.csv')

# Display all rows and columns in the DataFrame
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)

# Print the shape of the DataFrame and display the first few rows
print(df.shape)
df.head()

(7214, 53)


,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,juv_fel_count,decile_score,juv_misd_count,juv_other_count,priors_count,days_b_screening_arrest,c_jail_in,c_jail_out,c_case_number,c_offense_date,c_arrest_date,c_days_from_compas,c_charge_degree,c_charge_desc,is_recid,r_case_number,r_charge_degree,r_days_from_arrest,r_offense_date,r_charge_desc,r_jail_in,r_jail_out,violent_recid,is_violent_recid,vr_case_number,vr_charge_degree,vr_offense_date,vr_charge_desc,type_of_assessment,decile_score.1,score_text,screening_date,v_type_of_assessment,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,0,1,0,0,0,-1.0,2013-08-13 06:03:42,2013-08-14 05:41:20,13011352CF10A,2013-08-13,NaN,1.0,F,Aggravated Assault w/Firearm,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Risk of Recidivism,1,Low,2013-08-14,Risk of Violence,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,0,3,0,0,0,-1.0,2013-01-26 03:45:27,2013-02-05 05:36:53,13001275CF10A,2013-01-26,NaN,1.0,F,Felony Battery w/Prior Convict,1,13009779CF10A,(F3),NaN,2013-07-05,Felony Battery (Dom Strang),NaN,NaN,NaN,1,13009779CF10A,(F3),2013-07-05,Felony Battery (Dom Strang),Risk of Recidivism,3,Low,2013-01-27,Risk of Violence,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,0,4,0,1,4,-1.0,2013-04-13 04:58:34,2013-04-14 07:02:04,13005330CF10A,2013-04-13,NaN,1.0,F,Possession of Cocaine,1,13011511MM10A,(M1),0.0,2013-06-16,Driving Under The Influence,2013-06-16,2013-06-16,NaN,0,NaN,NaN,NaN,NaN,Risk of Recidivism,4,Low,2013-04-14,Risk of Violence,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,0,8,1,0,1,NaN,NaN,NaN,13000570CF10A,2013-01-12,NaN,1.0,F,Possession of Cannabis,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Risk of Recidivism,8,High,2013-01-13,Risk of Violence,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,0,1,0,0,2,NaN,NaN,NaN,12014130CF10A,NaN,2013-01-09,76.0,F,arrest case no charge,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Risk of Recidivism,1,Low,2013-03-26,Risk of Violence,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0


Trying to guage what columns should be dropped using the sum of .isnull() and other factors. Some like `is_recid` are considered **target leakage** because their values could change in the at the time the prediction is made. Also dropping identifiers such as `name` and `id`. 

In [4]:
print(f"Checking if 'is_recid' and 'two_year_recid' have mostly same values. Supports target leakage:\n{(df['is_recid'] == df['two_year_recid']).value_counts()}")
print(f"\nChecking if 'priors_count' and 'priors_count.1' have same values:\n{(df['priors_count'] == df['priors_count.1']).all()}")
print(f"\nChecking for missing values:\n{df.isnull().sum()}")

Checking if 'is_recid' and 'two_year_recid' have mostly same values. Supports target leakage:
True     6994
False     220
Name: count, dtype: int64

Checking if 'priors_count' and 'priors_count.1' have same values:
True

Checking for missing values:
id                            0
name                          0
first                         0
last                          0
compas_screening_date         0
sex                           0
dob                           0
age                           0
age_cat                       0
race                          0
juv_fel_count                 0
decile_score                  0
juv_misd_count                0
juv_other_count               0
priors_count                  0
days_b_screening_arrest     307
c_jail_in                   307
c_jail_out                  307
c_case_number                22
c_offense_date             1159
c_arrest_date              6077
c_days_from_compas           22
c_charge_degree               0
c_charge_desc 

I had to exclude `race` because of potential bias when training data. I'm using the same filters from the ProPublica COMPAS-analysis notebook. According to them they claim the following:

> "Not all of the rows are useable for the first round of analysis.
> 
> There are a number of reasons remove rows because of missing data:
> 
> If the charge date of a defendants Compas scored crime was not within 30 days from when the person was arrested, we assume that because of data quality reasons, that we do not have the right offense.
> We coded the recidivist flag -- is_recid -- to be -1 if we could not find a compas case at all.
> In a similar vein, ordinary traffic offenses -- those with a c_charge_degree of 'O' -- will not result in Jail time are removed (only two of them).
> We filtered the underlying data from Broward county to include only those rows representing people who had either recidivated in two years, or had at least two years outside of a > correctional facility.
>
> > Row-filtering criteria adapted from ProPublica's original COMPAS analysis (Larson et al., 2016), source: https://github.com/propublica/compas-analysis

Will include other features later such as `score_text` and `decile_score` for 2nd analysis.



In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Using same filters from propubica dataset to clean the data and remove leakage columns
df_clean = df[
    (df['days_b_screening_arrest'] <= 30) &
    (df['days_b_screening_arrest'] >= -30) &
    (df['is_recid'] != -1) &
    (df['c_charge_degree'] != "O") &
    (df['score_text'] != "N/A")
]

print(df_clean.shape)
leakage_cols = ['r_case_number', 'r_charge_degree', 'r_days_from_arrest', 
                 'r_offense_date', 'r_charge_desc', 'r_jail_in', 'r_jail_out',
                 'violent_recid', 'vr_case_number', 'vr_charge_degree', 
                 'vr_offense_date', 'vr_charge_desc', 'is_recid', 'priors_count.1',
                 'name', 'first', 'last', 'id']

df_clean = df_clean.drop(columns=leakage_cols)
print(df_clean.shape)

y = df_clean['two_year_recid']
feature_names = ['sex', 'age', 'priors_count', 'c_charge_degree']
X = df_clean[feature_names]

print(y.unique())
# Divide data into training and validation subsets
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)


(6172, 53)
(6172, 35)
[0 1]


## Baseline Model Performance

Trained a RandomForestClassifier (n_estimators=100) on `sex`, `age`, `priors_count`, 
and `c_charge_degree` (excluding `race` and COMPAS's own predictions to avoid 
circularity/leakage). Achieved **64.4% accuracy** on a held-out validation set.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score

object_cols = ['sex', 'c_charge_degree']

OH_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
OH_train_cols = pd.DataFrame(OH_encoder.fit_transform(train_X[['sex', 'c_charge_degree']]))
OH_valid_cols = pd.DataFrame(OH_encoder.transform(val_X[['sex', 'c_charge_degree']]))

OH_train_cols.index = train_X.index
OH_valid_cols.index = val_X.index

num_train_X = train_X.drop(object_cols, axis=1)
num_val_X = val_X.drop(object_cols, axis = 1)

OH_train_X = pd.concat([num_train_X, OH_train_cols], axis=1)
OH_val_X = pd.concat([num_val_X, OH_valid_cols], axis=1)

OH_train_X.columns = OH_train_X.columns.astype(str)
OH_val_X.columns = OH_val_X.columns.astype(str)

bias_model = RandomForestClassifier(n_estimators=100, random_state=1).fit(OH_train_X, train_y)
predictions = bias_model.predict(OH_val_X)

accuracy_score(val_y, predictions)

0.6435515230071289

Fairlearn

In [28]:
from fairlearn.metrics import MetricFrame, count, false_positive_rate, selection_rate
from sklearn.metrics import recall_score

race = df_clean.loc[val_X.index, 'race']
#Construct the function dict
metrics = {
    'tpr' : recall_score,
    'fpr' : false_positive_rate,
    'sel' : selection_rate,
    'count' : count
}

# Construct the MetricFrame
mf = MetricFrame(
    metrics = metrics,
    y_true= val_y,
    y_pred= predictions,
    sensitive_features= race
)

mf.by_group

,tpr,fpr,sel,count
race,,,,
African-American,0.602381,0.363158,0.488750,800.0
Asian,0.500000,0.000000,0.142857,7.0
Caucasian,0.455000,0.202492,0.299424,521.0
Hispanic,0.545455,0.273810,0.367188,128.0
Native American,1.000000,0.666667,0.800000,5.0
Other,0.535714,0.222222,0.329268,82.0
